# Norm Provenance Ledger v0.4 – пилот на 10 прогонах в Docent

Запускай ячейки сверху вниз по одной. Ячейки 1–4 ничего не отправляют в Docent. Сетевые записи начинаются только в ячейке 6, и только после того, как ты впишешь судью.

Понадобится: файл `norm_ledger_v04.zip` на телефоне (скачай его из чата в «Файлы») и ключ API Docent.

## 1. Установка SDK (версия закреплена)

In [ ]:
!pip -q install docent==0.1.87
import docent, importlib.metadata as im
print('docent', im.version('docent'))

## 2. Загрузка архива
После запуска нажми «Выбрать файлы» и выбери `norm_ledger_v04.zip`.

In [ ]:
from google.colab import files
import zipfile, os, pathlib
up = files.upload()
name = next(n for n in up if n.endswith('.zip'))
zipfile.ZipFile(name).extractall('/content')
ROOT = pathlib.Path('/content/norm_ledger_v04')
os.chdir(ROOT)
print('рабочая папка:', ROOT)
print(sorted(p.name for p in ROOT.iterdir()))

## 3. Офлайн-проверка (без сети)
Должно вывести 10 прогонов и 394 блока.

In [ ]:
!python protocol/docent_run.py dry-run

## 4. Ключ API
Способ 1 (надёжнее): иконка ключа слева → Secrets → добавить `DOCENT_API_KEY`, включить доступ для блокнота.
Способ 2: ячейка сама спросит ключ скрытым полем. Ключ не печатается и не сохраняется в блокноте.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['DOCENT_API_KEY'] = userdata.get('DOCENT_API_KEY')
    print('ключ взят из Secrets')
except Exception:
    from getpass import getpass
    os.environ['DOCENT_API_KEY'] = getpass('DOCENT_API_KEY: ')
    print('ключ введён вручную')
print('ключ задан:', bool(os.environ.get('DOCENT_API_KEY')))

## 5. Какие модели доступны
Первый список – судьи, доступные без собственного ключа провайдера, как они записаны в SDK. Второй – то, что сервер отдаёт для «readings» (может не совпадать со списком судей). Выбери одну модель и впиши её в ячейку 6 в виде `provider:model:effort`.

In [ ]:
from docent.sdk import Docent
from docent._llm_util.providers.preference_types import PUBLIC_PROVIDER_PREFERENCES
print('Судьи по умолчанию в SDK 0.1.87:')
for m in PUBLIC_PROVIDER_PREFERENCES.default_judge_models:
    print(f'  {m.provider}:{m.model_name}:{m.reasoning_effort}')
try:
    c = Docent(api_key=os.environ['DOCENT_API_KEY'])
    print('\nМодели на сервере (readings):')
    for m in c.get_reading_models():
        print('  ', {k: m.get(k) for k in ('provider','model_name','reasoning_effort','uses_byok')})
except Exception as e:
    print('список с сервера не получен:', type(e).__name__, str(e)[:200])

## 6. Запуск пилота v0.4 (сетевые записи)
Тот же судья, что в пилотах SWE-bench. Создаётся новая коллекция «v0.4 – pilot 10 (tau2 + MAST)» и новый файл состояния.

In [ ]:
JUDGE = 'openai:gpt-5.6-terra:medium'
!python protocol/docent_run.py launch --judge "$JUDGE" --label v0.4
from google.colab import files
files.download('docent_pilot_state_v0.4.json')

## 7. Ожидание результатов
Проверяет состояние раз в минуту до 30 минут. Можно перезапускать.

In [ ]:
import json, time
st = json.load(open('docent_pilot_state_v0.4.json'))
c = Docent(api_key=os.environ['DOCENT_API_KEY'])
for i in range(40):
    try:
        s = c.get_rubric_run_state(st['collection_id'], st['rubric_id'], include_failures=True)
        res = s.get('results') or []
        print(time.strftime('%H:%M:%S'), 'результатов:', len(res), 'из 10')
        if len(res) >= 10:
            print('готово'); break
    except Exception as e:
        print(time.strftime('%H:%M:%S'), 'сервер не ответил, повторю через минуту:', type(e).__name__)
    time.sleep(60)

## 8. Сырая выгрузка
Сохраняет ответ сервера целиком, с отказами. Этот файл пришли в чат, дальше нормализацию и метрики сделаю я.

In [ ]:
!python protocol/docent_run.py raw-export --label v0.4
files.download('raw_run_state_v0.4.json')

## Если Colab сбросился
Заново ячейки 1–4, затем загрузи сохранённый `docent_pilot_state.json` в `/content/norm_ledger_v04/` (через `files.upload()`) и сразу переходи к ячейкам 7–8. Ячейку 6 повторно не запускай.